In [ ]:
from __future__ import annotations

import astropy.units as u
import numpy as np
import xarray as xr

from radioskysims.domain.common.models import FrequencyRange

In [ ]:
def test_from_floats():
    fr = FrequencyRange.from_floats(100, 200, 10)
    assert fr.min == 100.0
    assert fr.max == 200.0
    assert fr.resolution == 10.0


def test_from_floats_with_unit():
    fr = FrequencyRange.from_floats(1e8, 2e8, 1e7, unit=u.Hz)
    assert np.isclose(fr.min, 100.0)
    assert np.isclose(fr.max, 200.0)
    assert np.isclose(fr.resolution, 10.0)


def test_from_array_without_units():
    arr = np.linspace(100, 200, 11)
    fr = FrequencyRange.from_array(arr)
    assert np.isclose(fr.min, 100.0)
    assert np.isclose(fr.max, 200.0)
    assert np.isclose(fr.resolution, 10.0)


def test_from_array_with_units():
    arr = [100 * u.MHz, 110 * u.MHz, 120 * u.MHz]
    fr = FrequencyRange.from_array(arr)
    assert fr.resolution == 10.0


def test_from_string():
    fr = FrequencyRange.from_string("100,200,10")
    assert fr.min == 100.0
    assert fr.max == 200.0
    assert fr.resolution == 10.0


def test_from_config():
    cfg = {"min": 100, "max": 200, "resolution": 10, "unit": "MHz"}
    fr = FrequencyRange.from_config(cfg)
    assert fr.min == 100.0
    assert fr.max == 200.0
    assert fr.resolution == 10.0


def test_default_resolution():
    fr = FrequencyRange.from_floats(100, 200)
    assert len(fr.value) == 1024
    assert np.isclose(fr.resolution, (200 - 100) / 1023)


def test_range_and_value():
    fr = FrequencyRange.from_floats(100, 200, 10)
    assert isinstance(fr.range, xr.DataArray)
    assert fr.range.attrs["units"] == "MHz"
    assert np.allclose(fr.value, np.arange(100, 201, 10))

In [3]:
arr = np.linspace(100, 200, 11)
fr = FrequencyRange.from_array(arr)
fr.min

100.0

In [4]:
test_from_floats()

In [5]:
test_from_floats_with_unit()

In [6]:
test_from_array_without_units()

In [7]:
test_from_array_with_units()

In [8]:
test_from_string()

In [9]:
test_from_config()

In [10]:
test_default_resolution()

In [11]:
test_range_and_value()

In [ ]:
import numpy as np
from astropy.coordinates import Angle

from radioskysims.domain.common.models import AngularRange


def test_from_degrees():
    ar = AngularRange.from_degrees(0, 90, 10)
    assert np.isclose(ar.min, 0)
    assert np.isclose(ar.max, np.pi / 2)
    assert ar.resolution > 0


def test_from_hours():
    ar = AngularRange.from_hours(0, 6, 1)
    assert np.isclose(ar.max, Angle(6, unit=u.hourangle).rad)
    assert ar.resolution > 0


def test_from_array_degrees():
    ar = AngularRange.from_array([0, 30, 60, 90] * u.deg)
    assert np.isclose(ar.resolution, np.radians(30))


def test_from_array_hours():
    ar = AngularRange.from_array([0, 3, 6] * u.hourangle)
    expected = Angle(3, unit=u.hourangle).rad
    assert np.isclose(ar.resolution, expected)


def test_from_string():
    ar = AngularRange.from_string("0,90,30", unit=u.deg)
    assert ar.max == np.pi / 2


def test_from_config():
    cfg = {"min": 0, "max": 180, "resolution": 30, "unit": "deg"}
    ar = AngularRange.from_config(cfg)
    assert np.isclose(ar.max, np.pi)


def test_default_resolution():
    ar = AngularRange.from_degrees(0, 90)
    expected = np.radians(1)
    assert np.isclose(ar.resolution, expected)


def test_range_xarray():
    ar = AngularRange.from_degrees(0, 90, 30)
    xr_data = ar.range
    assert "angle" in xr_data.dims
    assert xr_data.attrs["units"] == "rad"


def test_repr():
    ar = AngularRange.from_degrees(0, 90, 30)
    assert "AngularRange" in repr(ar)
    assert "°" in repr(ar)


In [13]:
test_from_degrees()

In [14]:
test_from_hours()
test_from_array_degrees()
test_from_array_hours()
test_from_string()
test_from_config()
test_default_resolution()
test_range_xarray()
test_repr()

In [ ]:
import numpy as np
import pandas as pd
from astropy.time import Time

from radioskysims.domain.common.models import TimeRange


def test_from_iso():
    tr = TimeRange.from_iso("2020-01-01T00:00:00", "2020-01-01T01:00:00", resolution=60)
    assert isinstance(tr, TimeRange)
    assert np.isclose(tr.resolution, 60.0)


def test_from_jd():
    t0 = Time("2020-01-01T00:00:00").jd
    t1 = Time("2020-01-01T01:00:00").jd
    tr = TimeRange.from_jd(t0, t1, resolution=120)
    assert tr.min < tr.max
    assert np.isclose(tr.resolution, 120)


def test_from_astropy():
    t0 = Time("2020-01-01T00:00:00")
    t1 = Time("2020-01-01T02:00:00")
    tr = TimeRange.from_astropy(t0, t1, resolution=1800)
    assert tr.start_tai < tr.end_tai
    assert np.isclose(tr.resolution, 1800)


def test_from_config():
    cfg = {
        "start": "2020-01-01T00:00:00",
        "end": "2020-01-01T01:00:00",
        "resolution": 60,
        "unit": "s",
    }
    tr = TimeRange.from_config(cfg)
    assert isinstance(tr, TimeRange)
    assert np.isclose(tr.resolution, 60)


def test_default_resolution():
    t0 = Time("2020-01-01T00:00:00")
    t1 = Time("2020-01-01T00:10:00")
    tr = TimeRange.from_astropy(t0, t1)
    assert np.isclose(tr.resolution, 60.0)


def test_value_array():
    tr = TimeRange.from_iso("2020-01-01T00:00:00", "2020-01-01T00:05:00", resolution=60)
    values = tr.value
    assert isinstance(values, np.ndarray)
    assert len(values) == 6  # inclui o endpoint


def test_to_astropy():
    tr = TimeRange.from_iso("2020-01-01T00:00:00", "2020-01-01T00:03:00", resolution=60)
    times = tr.to_astropy()
    assert isinstance(times, Time)
    assert times.shape[0] == 4


def test_to_numpy():
    tr = TimeRange.from_iso("2020-01-01T00:00:00", "2020-01-01T00:04:00", resolution=60)
    jd = tr.to_numpy()
    assert isinstance(jd, np.ndarray)
    assert jd.shape[0] == 5


def test_to_pandas():
    tr = TimeRange.from_iso(
        "2020-01-01T00:00:00", "2020-01-01T00:10:00", resolution=300
    )
    dt_index = tr.to_pandas()
    assert isinstance(dt_index, pd.DatetimeIndex)
    assert len(dt_index) == 3


def test_range_xarray():
    tr = TimeRange.from_iso("2020-01-01T00:00:00", "2020-01-01T00:04:00", resolution=60)
    xr_data = tr.range
    assert isinstance(xr_data, xr.DataArray)
    assert "time" in xr_data.dims
    assert xr_data.attrs["units"] == "TAI seconds"


def test_repr_output():
    tr = TimeRange.from_iso("2020-01-01T00:00:00", "2020-01-01T01:00:00", resolution=60)
    rep = repr(tr)
    assert "TimeRange" in rep
    assert "resolution=" in rep


In [16]:
test_from_iso()
test_from_jd()
test_from_astropy()
test_from_config()
test_default_resolution()
test_value_array()
test_to_astropy()

In [17]:
tr = TimeRange.from_iso("2020-01-01T00:00:00", "2020-01-01T00:10:00", resolution=300)
dt_index = tr.to_pandas()

In [18]:
pd.to_datetime(tr.to_astropy().to_datetime())

DatetimeIndex([       '2020-01-01 00:00:00', '2020-01-01 00:04:59.999982',
               '2020-01-01 00:10:00.000004'],
              dtype='datetime64[ns]', freq=None)